# UnHateMemeDL — Model Comparison Analysis
Detection and mitigation results across all evaluated VLMs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#16213e',
    'legend.edgecolor': '#444',
})

PALETTE   = ['#e94560', '#0f3460', '#533483', '#05c46b']
EVAL_ROOT = Path('full_eval')
print('Setup done.')

## 1 — Load data

In [ ]:
summary = pd.read_csv(EVAL_ROOT / 'summary.csv')
summary['model_label'] = summary['vlm_name'].str.split('/').str[-1]

det_dfs = {}
mit_dfs = {}
for _, row in summary.iterrows():
    slug = row['model_slug']
    det_path = EVAL_ROOT / slug / 'detection_predictions.csv'
    mit_path = EVAL_ROOT / slug / 'mitigation_results.csv'
    if det_path.exists():
        det_dfs[row['model_label']] = pd.read_csv(det_path)
    if mit_path.exists():
        mit_dfs[row['model_label']] = pd.read_csv(mit_path)

models = summary['model_label'].tolist()
print(f'{len(models)} models loaded:', models)

## 2 — Bootstrap confidence intervals

For each model we resample its predictions **1000 times** (with replacement) and recompute every metric.
The 95% CI is the interval between the 2.5th and 97.5th percentile of those 1000 values.
This gives error bars without any additional inference.

In [ ]:
def bootstrap_det(df, n_boot=1000, ci=95, seed=42):
    """Compute bootstrap CIs for detection metrics from a per-image predictions CSV."""
    rng  = np.random.default_rng(seed)
    valid = df[df['prob_pred'].notna() & (df['prob_pred'] != '') &
               (df['error'].isna() | (df['error'] == ''))].copy()
    valid['prob_pred']  = valid['prob_pred'].astype(float)
    valid['label_pred'] = valid['label_pred'].astype(int)
    valid['label_true'] = valid['label_true'].astype(int)

    yt = valid['label_true'].values
    yp = valid['prob_pred'].values
    yh = valid['label_pred'].values
    n  = len(yt)

    metric_fns = {
        'auroc':              lambda t, p, h: roc_auc_score(t, p),
        'macro_f1':           lambda t, p, h: f1_score(t, h, average='macro', zero_division=0),
        'accuracy':           lambda t, p, h: accuracy_score(t, h),
        'precision_hateful':  lambda t, p, h: precision_score(t, h, pos_label=1, zero_division=0),
        'recall_hateful':     lambda t, p, h: recall_score(t, h, pos_label=1, zero_division=0),
        'f1_hateful':         lambda t, p, h: f1_score(t, h, pos_label=1, zero_division=0),
    }

    boot = {k: [] for k in metric_fns}
    lo_p = (100 - ci) / 2
    hi_p = 100 - lo_p

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        for k, fn in metric_fns.items():
            try:
                boot[k].append(fn(yt[idx], yp[idx], yh[idx]))
            except Exception:
                pass

    result = {}
    for k, vals in boot.items():
        arr = np.array(vals)
        mean = arr.mean()
        result[k] = {
            'mean': mean,
            'lo':   mean - np.percentile(arr, lo_p),   # distance below
            'hi':   np.percentile(arr, hi_p) - mean,   # distance above
        }
    return result


def bootstrap_col(series, n_boot=1000, ci=95, seed=42):
    """Compute bootstrap CI for the mean of a numeric series."""
    rng  = np.random.default_rng(seed)
    vals = pd.to_numeric(series, errors='coerce').dropna().values
    if len(vals) == 0:
        return {'mean': np.nan, 'lo': 0, 'hi': 0}
    lo_p = (100 - ci) / 2
    hi_p = 100 - lo_p
    boot = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    arr  = np.array(boot)
    mean = vals.mean()
    return {'mean': mean, 'lo': mean - np.percentile(arr, lo_p), 'hi': np.percentile(arr, hi_p) - mean}


# --- Run bootstrap for all models ---
print('Computing bootstrap CIs (1000 resamples per model) …')
det_boot = {}
for model, df in det_dfs.items():
    det_boot[model] = bootstrap_det(df)
    print(f'  {model}: AUROC = {det_boot[model]["auroc"]["mean"]:.3f} '
          f'[±{(det_boot[model]["auroc"]["lo"]+det_boot[model]["auroc"]["hi"])/2:.3f}]')

print('Done.')

## 3 — Detection: metrics comparison with 95% CI error bars

In [ ]:
metric_keys = ['auroc', 'macro_f1', 'accuracy', 'precision_hateful', 'recall_hateful', 'f1_hateful']
metric_lbls = ['AUROC', 'Macro F1', 'Accuracy', 'Precision\n(hateful)', 'Recall\n(hateful)', 'F1\n(hateful)']

x     = np.arange(len(metric_keys))
width = 0.8 / len(models)

fig, ax = plt.subplots(figsize=(14, 5))
for i, model in enumerate(det_boot):
    ci_data = det_boot[model]
    means   = [ci_data[k]['mean'] for k in metric_keys]
    err_lo  = [ci_data[k]['lo']   for k in metric_keys]
    err_hi  = [ci_data[k]['hi']   for k in metric_keys]
    offset  = x + i * width - (len(det_boot)-1)*width/2

    bars = ax.bar(offset, means, width=width,
                  label=model, color=PALETTE[i % len(PALETTE)], alpha=0.85)
    ax.errorbar(offset, means, yerr=[err_lo, err_hi],
                fmt='none', color='white', capsize=3, linewidth=1.2)
    for bar, v in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f'{v:.2f}', ha='center', va='bottom', fontsize=6.5, color='white')

ax.set_xticks(x)
ax.set_xticklabels(metric_lbls)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Detection Metrics by Model  (error bars = 95% bootstrap CI)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'detection_metrics.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 4 — Detection: confusion matrices

In [ ]:
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, summary.iterrows()):
    cm = np.array([[row['tn'], row['fp']],
                   [row['fn'], row['tp']]], dtype=int)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=False, fmt='', ax=ax,
                cmap='RdYlGn', vmin=0, vmax=1,
                linewidths=2, linecolor='#1a1a2e', cbar=False)
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.35, str(cm[i,j]),
                    ha='center', va='center', fontsize=16, fontweight='bold', color='white')
            ax.text(j+0.5, i+0.65, f'{cm_norm[i,j]:.0%}',
                    ha='center', va='center', fontsize=10, color='white')

    ax.set_xticklabels(['Pred: Non-hateful', 'Pred: Hateful'], rotation=15, ha='right')
    ax.set_yticklabels(['True: Non-hateful', 'True: Hateful'], rotation=0)
    ax.set_title(row['model_label'], fontsize=10, fontweight='bold', pad=8)

fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'confusion_matrices.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 5 — Detection: predicted probability distributions

In [ ]:
fig, axes = plt.subplots(1, len(det_dfs), figsize=(5*len(det_dfs), 4), sharey=True)
if len(det_dfs) == 1:
    axes = [axes]

for ax, (model, df) in zip(axes, det_dfs.items()):
    valid = df[df['prob_pred'].notna() & (df['prob_pred'] != '')].copy()
    valid['prob_pred'] = valid['prob_pred'].astype(float)
    for label, grp in valid.groupby('label_true'):
        color = '#e94560' if label == 1 else '#05c46b'
        name  = 'Hateful' if label == 1 else 'Non-hateful'
        ax.hist(grp['prob_pred'], bins=20, range=(0,1),
                alpha=0.65, color=color, label=name, density=True)
    ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold')
    ax.set_title(model, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted probability')
    ax.legend(fontsize=8)
    ax.grid(axis='y')

axes[0].set_ylabel('Density')
fig.suptitle('Predicted Probability Distributions (by true label)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'prob_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 6 — Detection: false positives & false negatives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title, color in zip(
    axes,
    ['fp', 'fn'],
    ['False Positives (non-hateful → predicted hateful)',
     'False Negatives (hateful → predicted non-hateful)'],
    ['#f39c12', '#e94560']
):
    vals   = summary[key].tolist()
    labels = summary['model_label'].tolist()
    bars = ax.barh(labels, vals, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(v+1, bar.get_y()+bar.get_height()/2,
                str(int(v)), va='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Count')
    ax.grid(axis='x')

plt.suptitle('Error Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'error_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 7 — Detection: inference time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
labels = summary['model_label'].tolist()
means  = summary['det_mean_s'].tolist()
stds   = summary['det_std_s'].tolist()
bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(labels))]

bars = ax.bar(labels, means, yerr=stds, capsize=5, color=bar_colors, alpha=0.85,
              error_kw={'ecolor': 'white', 'linewidth': 1.5})
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(stds)*0.05,
            f'{m:.2f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Time per image (seconds)')
ax.set_title('Detection Inference Time  (mean ± std across images)', fontsize=12, fontweight='bold')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'detection_time.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 8 — Mitigation: toxicity reduction with 95% CI error bars

In [ ]:
mit_summary = summary[summary['n_mitigated'].notna() & (summary['n_mitigated'] > 0)].copy()

# Bootstrap CI on prob_after and pct_nonhateful per model
mit_boot = {}
for _, row in mit_summary.iterrows():
    model = row['model_label']
    if model not in mit_dfs:
        continue
    df_h = mit_dfs[model]
    df_h = df_h[(df_h['label_true'] == 1) &
                df_h['prob_after'].notna() & (df_h['prob_after'] != '') &
                df_h['prob_before'].notna() & (df_h['prob_before'] != '')].copy()
    df_h['prob_after']  = pd.to_numeric(df_h['prob_after'],  errors='coerce')
    df_h['prob_before'] = pd.to_numeric(df_h['prob_before'], errors='coerce')
    df_h = df_h.dropna(subset=['prob_after', 'prob_before'])

    mit_boot[model] = {
        'prob_after':         bootstrap_col(df_h['prob_after']),
        'pct_nonhateful':     bootstrap_col((df_h['prob_after'] < 0.5).astype(float) * 100),
        'bertscore_f1':       bootstrap_col(df_h.get('bertscore_f1', pd.Series(dtype=float))),
        'clip_score':         bootstrap_col(df_h.get('clip_score',   pd.Series(dtype=float))),
        'ssim':               bootstrap_col(df_h.get('ssim',         pd.Series(dtype=float))),
        'mps':                bootstrap_col(df_h.get('mps',          pd.Series(dtype=float))),
    }

print('Mitigation bootstrap done.')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# prob_before vs prob_after
ax = axes[0]
x  = np.arange(len(mit_summary))
w  = 0.35
bars_b = ax.bar(x - w/2, mit_summary['mean_prob_before'], w, label='Before', color='#e94560', alpha=0.85)
bars_a = ax.bar(x + w/2, mit_summary['mean_prob_after'],  w, label='After',  color='#05c46b', alpha=0.85)

# CI on prob_after only (prob_before is fixed from detection)
for j, (_, row) in enumerate(mit_summary.iterrows()):
    m = row['model_label']
    if m in mit_boot:
        ci = mit_boot[m]['prob_after']
        ax.errorbar(x[j] + w/2, ci['mean'], yerr=[[ci['lo']], [ci['hi']]],
                    fmt='none', color='white', capsize=4, linewidth=1.3)

for bars in [bars_b, bars_a]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.2f}', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(mit_summary['model_label'], rotation=10, ha='right')
ax.set_ylabel('Mean predicted probability')
ax.set_title('Mean Hatefulness Score Before vs After Mitigation', fontweight='bold')
ax.axhline(0.5, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.legend()
ax.grid(axis='y')

# % non-hateful after with CI
ax = axes[1]
bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(mit_summary))]
bars = ax.bar(mit_summary['model_label'], mit_summary['pct_nonhateful_after'],
              color=bar_colors, alpha=0.85)
for j, (_, row) in enumerate(mit_summary.iterrows()):
    m = row['model_label']
    if m in mit_boot:
        ci = mit_boot[m]['pct_nonhateful']
        ax.errorbar(j, ci['mean'], yerr=[[ci['lo']], [ci['hi']]],
                    fmt='none', color='white', capsize=4, linewidth=1.3)
for bar, v in zip(bars, mit_summary['pct_nonhateful_after']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_xticklabels(mit_summary['model_label'], rotation=10, ha='right')
ax.set_ylabel('% images with prob_after < 0.5')
ax.set_title('% Successfully Mitigated  (error bars = 95% CI)', fontweight='bold')
ax.set_ylim(0, 115)
ax.grid(axis='y')

plt.suptitle('Mitigation — Toxicity Reduction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'mitigation_toxicity.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 9 — Mitigation: prob_after distributions

In [ ]:
valid_mit = {m: df for m, df in mit_dfs.items() if 'prob_after' in df.columns}
fig, axes = plt.subplots(1, len(valid_mit), figsize=(5*len(valid_mit), 4), sharey=True)
if len(valid_mit) == 1:
    axes = [axes]

for ax, (model, df) in zip(axes, valid_mit.items()):
    df_h = df[(df['label_true']==1) & df['prob_after'].notna() & (df['prob_after']!='')].copy()
    df_h['prob_after']  = df_h['prob_after'].astype(float)
    df_h['prob_before'] = pd.to_numeric(df_h['prob_before'], errors='coerce')
    ax.hist(df_h['prob_before'].dropna(), bins=20, range=(0,1),
            alpha=0.5, color='#e94560', label='Before', density=True)
    ax.hist(df_h['prob_after'], bins=20, range=(0,1),
            alpha=0.65, color='#05c46b', label='After', density=True)
    ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_title(model, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted probability')
    ax.legend(fontsize=8)
    ax.grid(axis='y')

axes[0].set_ylabel('Density')
fig.suptitle('Probability Distribution Before vs After Mitigation (hateful images only)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'mitigation_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 10 — Mitigation: content preservation with 95% CI error bars

In [ ]:
pres_keys = ['bertscore_f1', 'clip_score', 'ssim', 'mps']
pres_lbls = ['BERTScore F1', 'CLIPScore', 'SSIM', 'MPS']

x     = np.arange(len(pres_keys))
width = 0.8 / len(mit_boot)

fig, ax = plt.subplots(figsize=(11, 5))
for i, (model, ci_data) in enumerate(mit_boot.items()):
    means  = [ci_data[k]['mean'] for k in pres_keys]
    err_lo = [ci_data[k]['lo']   for k in pres_keys]
    err_hi = [ci_data[k]['hi']   for k in pres_keys]
    offset = x + i*width - (len(mit_boot)-1)*width/2

    bars = ax.bar(offset, means, width=width, label=model,
                  color=PALETTE[i % len(PALETTE)], alpha=0.85)
    ax.errorbar(offset, means, yerr=[err_lo, err_hi],
                fmt='none', color='white', capsize=3, linewidth=1.2)
    for bar, v in zip(bars, means):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, color='white')

ax.set_xticks(x)
ax.set_xticklabels(pres_lbls, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Content Preservation Metrics  (error bars = 95% bootstrap CI)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'mitigation_preservation.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 11 — Mitigation: inference time breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(mit_summary))
w = 0.35

ax.bar(x-w/2, mit_summary['prompt_mean_s'],    w, label='VLM prompt gen',
       color='#533483', alpha=0.9,
       yerr=mit_summary['prompt_std_s'], capsize=4, error_kw={'ecolor':'white','linewidth':1})
ax.bar(x+w/2, mit_summary['diffusion_mean_s'], w, label='Diffusion',
       color='#0f3460', alpha=0.9,
       yerr=mit_summary['diffusion_std_s'], capsize=4, error_kw={'ecolor':'white','linewidth':1})

ax.set_xticks(x)
ax.set_xticklabels(mit_summary['model_label'], rotation=10, ha='right')
ax.set_ylabel('Time per image (seconds)')
ax.set_title('Mitigation Inference Time: VLM prompt gen vs Diffusion  (mean ± std)', fontweight='bold')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'mitigation_time.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 12 — Pareto: toxicity reduction vs content preservation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, (model, ci_data) in enumerate(mit_boot.items()):
    x_val = ci_data['bertscore_f1']['mean']
    y_val = ci_data['pct_nonhateful']['mean']
    x_err = [[ci_data['bertscore_f1']['lo']], [ci_data['bertscore_f1']['hi']]]
    y_err = [[ci_data['pct_nonhateful']['lo']], [ci_data['pct_nonhateful']['hi']]]
    if np.isnan(x_val) or np.isnan(y_val):
        continue
    ax.errorbar(x_val, y_val, xerr=x_err, yerr=y_err,
                fmt='o', ms=12, color=PALETTE[i % len(PALETTE)],
                ecolor='white', capsize=4, linewidth=1.2,
                markeredgecolor='white', markeredgewidth=1)
    ax.annotate(model, (x_val, y_val),
                textcoords='offset points', xytext=(8, 4),
                fontsize=9, color='white')

ax.set_xlabel('BERTScore F1  (content preservation →)', fontsize=11)
ax.set_ylabel('% Non-hateful after mitigation  (toxicity reduction →)', fontsize=11)
ax.set_title('Pareto: Toxicity Reduction vs Content Preservation\n(top-right corner = best, error bars = 95% CI)',
             fontweight='bold')
ax.grid(True)
plt.tight_layout()
plt.savefig(EVAL_ROOT / 'pareto.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 13 — Full summary table

In [ ]:
display_cols = [
    'model_label',
    'auroc', 'macro_f1', 'accuracy', 'fp', 'fn', 'det_mean_s',
    'pct_nonhateful_after', 'mean_bertscore_f1', 'mean_clip_score', 'mean_ssim',
    'prompt_mean_s', 'diffusion_mean_s',
]
table = summary[display_cols].set_index('model_label')
table.columns = [
    'AUROC', 'Macro F1', 'Accuracy', 'FP', 'FN', 'Det time (s)',
    '% mitigated', 'BERTScore', 'CLIPScore', 'SSIM',
    'Prompt (s)', 'Diffusion (s)',
]
table.style \
    .highlight_max(axis=0, subset=['AUROC','Macro F1','Accuracy','% mitigated','BERTScore','CLIPScore','SSIM'], color='#05c46b') \
    .highlight_min(axis=0, subset=['FP','FN','Det time (s)','Prompt (s)','Diffusion (s)'], color='#05c46b') \
    .format(precision=3)